# From Noise Prediction to ODE Solving in Diffusion Models

- Where from comes ODE?


## 1. What is the function we're dealing with?

The function is the **denoising process** that transforms pure noise into an image:
- **Input**: $x_T$ (pure Gaussian noise)
- **Output**: $x_0$ (clean image)
- **Function**: $x(t)$ where $t$ goes from $T$ (noisy) to $0$ (clean)

Think of $x(t)$ as a trajectory in high-dimensional space that smoothly transforms noise into an image.

## 2. How it looks WITHOUT ODE (discrete DDPM)

In the original DDPM, we have discrete steps:
```
x_T → x_{T-1} → x_{T-2} → ... → x_1 → x_0
```

At each step, we predict noise $\epsilon_\theta(x_t, t)$ and update:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t, t)\right) + \sigma_t \epsilon = f(x_t,t)$$

$ f(x_t,t) $ - some function of $x_t,t$

This is a **discrete iterative process** - we jump from one timestep to another in fixed intervals.

## 3. Where does the derivative come from?

Here's the key insight: **What if we make the timesteps infinitesimally small?**

### Step-by-step transformation:

**Step 1**: Rewrite the discrete update as a change:
$$x_{t-1} - x_t = f(x_t,t)- x_t $$

in this particular case $\Delta t = 1$
and we can rewrite equation in form:

$$x_{t-\Delta t} - x_t = f(x_t,t, \Delta t)- x_t $$



**Step 2**: Divide both sides by the timestep $\Delta t $:
$$\frac{x_{t-1} - x_t}{\Delta t} = \frac{x_t -f(x_t,t)}{\Delta t}$$

**Step 3**: As $\Delta t \to 0$, this becomes a derivative:
Since ( t ) decreases, flip the sign:
$$\frac{dx}{dt} = - \lim_{\Delta t \to 0} \frac{x_{t-\Delta t} - x_t}{\Delta t} = -f(x_t,t)$$

it's is ODE


**Step 4**: The function $f(x_t, t)$ is derived from the noise prediction:
$$f(x_t, t) = -\frac{\epsilon_\theta(x_t, t)}{\sqrt{1-\bar{\alpha}_t}}$$
(simplified form; exact expression depends on the noise schedule)

## Concrete Example

Let's see this with numbers:

**Discrete version** (DDPM):
- At $t=50$: $x_{50}$ = noisy image
- Predict noise: $\epsilon_\theta(x_{50}, 50)$
- Update: $x_{49} = x_{50} - \text{[scaled noise]}$

**Continuous version** (ODE):
- At time $t$: $x(t)$ = noisy image
- Rate of change: $\frac{dx}{dt} = -\epsilon_\theta(x(t), t) / \text{[scaling factor]}$
- This describes how $x$ changes continuously over time

## Why make this jump to ODE?

1. **Mathematical elegance**: ODEs have well-studied properties
2. **Flexibility**: Can use any ODE solver (not just Euler steps)
3. **Speed**: Advanced ODE solvers can skip steps while maintaining accuracy
4. **Deterministic**: No random noise added during generation

## Summary

The jump from "predicting noise" to "solving ODE" happens when we:
1. View the denoising process as a continuous trajectory $x(t)$
2. Express the discrete updates as rates of change
3. Take the limit as timesteps become infinitesimal
4. Get a differential equation: $\frac{dx}{dt} = -g(\epsilon_\theta(x, t), t)$

The noise prediction $\epsilon_\theta$ is still there - it's just wrapped inside the ODE's right-hand side function!


Rectified Flow


$\frac{dZ_t}{dt} = v(Z_t, t)$


$\frac{dZ_t}{dt} \approx \frac{\Delta Z}{\Delta t}$

$\frac{Z_{t_{i-1}} - Z_{t_i}}{t_{i-1} - t_i} \approx \frac{dZ}{dt} \bigg|_{t=t_i} $

$ \frac{Z_{t_{i-1}} - Z_{t_i}}{t_{i-1} - t_i} \approx v(Z_{t_i}, t_i) $

# Losses


## DDPM / DDIM (same trained model)

Forward noising:

$ x_t \;=\; \sqrt{\bar{\alpha}_t}\,x_0 \;+\; \sqrt{1-\bar{\alpha}_t}\,\varepsilon,
\qquad \varepsilon\sim\mathcal{N}(0,I).$

Noise-prediction objective:

$
\mathcal{L}_{\text{DDPM}}
=\mathbb{E}_{\,t,\,x_0,\,\varepsilon}\!\left[
w_t\,\big\|\varepsilon_\theta(x_t,t)-\varepsilon\big\|_2^{2}
\right] $

---



## Rectified Flow (Flow Matching)

Sample $ (x_0\sim p_{\text{data}},\;x_1\sim\mathcal{N}(0,I),\;t\sim\mathcal{U}[0,1])$ and define the straight bridge:

$ x_t \;=\; (1-t)\,x_1 \;+\; t\,x_0. $



Velocity-matching objective:

$
\mathcal{L}_{\text{RF}}
=\mathbb{E}_{\,x_0,\,x_1,\,t}\!\left[
\big\|\,v_\theta\!\big(x(t),t\big)\;-\;(x_0-x_1)\big\|_2^{2}
\right]
 $



$
\mathcal{L}_{\text{RF}}
=\mathbb{E}_{\,x_0,\,x_1,\,t}\!\left[
w(t)\,\big\|\,v_\theta\!\big(x(t),t\big)\;-\;(x_0-x_1)\big\|_2^{2}
\right]
 $

(Equivalent conditional form often used:)
$$
\mathcal{L}
=\mathbb{E}\!\left[
w(t)\,\big\|\,v_\theta(x,t)\;-\;\tfrac{x_0-x}{1-t}\big\|_2^{2}
\right],
\qquad x=(1-t)x_1+t x_0.
$$

[Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow](https://arxiv.org/abs/2209.03003)




Sampling

$x_{t-1} = x_t - \Delta t \cdot v_\theta(x_t, t)$


## Score matching


$\ score(x) = \nabla_x \log p(x) = \varepsilon$


$\mathcal{L} = \mathbb{E}_{x_0, \varepsilon, t} \left[ \left\| \varepsilon - \varepsilon_\theta(x_t, t) \right\|^2 \right]$

$\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$

## Common Sampler
*step 0:*

$ \varepsilon \sim \mathcal{N}(0, I)  \\
x_0 = \varepsilon
$

*step 1:*

$
\varepsilon_{pred} = model(x) \\
\hat{x_1} = x_0 - \varepsilon_{pred} \\
\varepsilon_1 = c (\sim \mathcal{N}(0, I)) \\
 x_1 = \hat{x_1} + \varepsilon_1
$

next step


$ x_{next} = a \, x_{curr} + b\, pred + c\,\varepsilon $



#Langevin dynamics


$ \begin{equation}
x_{\text{new}} = x + \frac{\eta}{2} \, \nabla_x \log p(x) + \sqrt{\eta} \, \varepsilon,
\end{equation} $


where $ \eta > 0 $ is a small step size and


 $ \varepsilon \sim \mathcal{N}(0, I)$

 is standard Gaussian noise.

#DDPM

$ x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \, \varepsilon_\theta(x_t, t) \right) + \sigma_t \, \varepsilon_t $




# Optimal transport theory

you’re learning a deterministic transport map between two distributions
t=0⇒noise,t=1⇒data.

flow direction (from random to structured)


[Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747)
[Scaling Rectified Flow Transformers for High-Resolution Image Synthesis](https://arxiv.org/pdf/2403.03206)
